# NPS Open Climate Data — Pipeline

Run this notebook in [Google Colab](https://colab.research.google.com) to generate climate data for all 63 US National Parks.

**Two paths:**
| | Path A | Path B |
|---|---|---|
| **Data** | Real (DAYMET + ERA5 via Earth Engine) | Synthetic demo data |
| **Requires** | Google Earth Engine project | Nothing extra |
| **Time** | ~2–4 hrs for all 63 parks | ~10 seconds |
| **Use for** | Publishing real results | Testing the site pipeline |

Run **Setup** (step 1), then skip to **Path A** or **Path B**, then continue from **Step 3** onward.

## 1. Setup

In [ ]:
# Clone the repo and install dependencies
import os, subprocess

REPO = "https://github.com/anniebritton/NPS-Open-Climate-Data.git"
REPO_DIR = "NPS-Open-Climate-Data"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO], check=True)

os.chdir(REPO_DIR)

subprocess.run(
    ["pip", "install", "-e", ".", "numpy", "pyarrow", "requests", "--quiet"],
    check=True,
)
print("Setup complete.")

---
## Path A — Real Earth Engine export

Skip to **Path B** if you just want to test the pipeline with synthetic data.

### A1. Authenticate with Earth Engine

You need a [Google Cloud project with the Earth Engine API enabled](https://developers.google.com/earth-engine/guides/access). Replace `your-project-id` below.

In [ ]:
import ee

GCP_PROJECT = "your-project-id"  # <-- change this

ee.Authenticate()        # opens a browser tab the first time
ee.Initialize(project=GCP_PROJECT)
print("Earth Engine initialized.")

### A2. Export climate data

The full 1980–present export for all 63 parks runs for **2–4 hours**. Colab sessions can disconnect after 90 minutes of inactivity, so it's safer to export a **subset of parks** at a time using the `SLUGS` list below.

Find all park slugs in [`nps_climate_data/parks.py`](nps_climate_data/parks.py) or leave `SLUGS = None` to export all parks at once.

In [ ]:
import subprocess

# Leave SLUGS as None to export all 63 parks, or list a subset:
# SLUGS = ["yellowstone", "glacier", "grand-canyon"]
SLUGS = None

cmd = [
    "python", "scripts/01_export_all_parks.py",
    "--start", "1980-01-01",
]
if SLUGS:
    cmd += ["--slugs"] + SLUGS

subprocess.run(cmd, check=True)
print("Export complete. Raw files written to data/raw/")

---
## Path B — Synthetic demo data

Generates plausible but synthetic climate series for all 63 parks in about 10 seconds — no Earth Engine required. Good for testing the full pipeline before running the real export.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/03_generate_demo_data.py"], check=True)
print("Demo data written to data/raw/")

---
## 3. Build site JSON summaries

Aggregates the raw daily series into annual/seasonal summaries, runs Mann–Kendall trend tests and Theil–Sen slopes, and writes per-park JSON files used by the Astro site.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/02_build_site_data.py"], check=True)
print("Site JSON written to data/site/")

## 4. Fetch park boundaries

Tries to pull real park polygons from the USGS PAD-US web service, then fills any gaps with circle approximations.

In [ ]:
import subprocess

# Step 1: real boundaries from USGS (may fail on network restrictions)
result = subprocess.run(["python", "scripts/06_fetch_padus_boundaries.py"])
if result.returncode != 0:
    print("USGS fetch failed — will use circle approximations for all parks.")

# Step 2: fill remaining gaps with circle approximations
subprocess.run(["python", "scripts/05_generate_boundaries.py"], check=True)
print("Boundaries written to site/public/data/boundaries/")

## 5. Copy data into the site's public folder

Copies the generated JSON summaries into `site/public/data/` and gzip-compresses the raw CSVs for web download links.

In [ ]:
import gzip, os, shutil
from pathlib import Path

# JSON summaries
pub = Path("site/public/data")
pub.mkdir(parents=True, exist_ok=True)
(pub / "parks").mkdir(exist_ok=True)

shutil.copy("data/site/parks.json", pub / "parks.json")
for f in Path("data/site/parks").glob("*.json"):
    shutil.copy(f, pub / "parks" / f.name)

# Gzip raw CSVs
raw_src = Path("data/raw")
raw_dst = pub / "raw"
raw_dst.mkdir(exist_ok=True)
for slug_dir in raw_src.iterdir():
    if not slug_dir.is_dir():
        continue
    out_dir = raw_dst / slug_dir.name
    out_dir.mkdir(exist_ok=True)
    for csv in slug_dir.glob("*.csv"):
        with open(csv, "rb") as f_in, gzip.open(out_dir / (csv.name + ".gz"), "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

print("Data copied to site/public/data/")
print(f"  Parks JSON: {len(list((pub/'parks').glob('*.json')))} files")
print(f"  Raw CSVs:   {len(list(raw_dst.rglob('*.gz')))} gzipped files")

## 6. Download outputs

Downloads the processed data as a zip. Unzip and copy `data/site/` into `site/public/data/` in your local repo checkout to update the site.

In [ ]:
import shutil
from google.colab import files

# Zip everything needed to update the site
shutil.make_archive("/content/nps_climate_outputs", "zip", ".", "data/site")
files.download("/content/nps_climate_outputs.zip")
print("Download started.")
print("Unzip and place the contents at site/public/data/ in your local repo.")

---
## Optional: preview the Astro site locally

After downloading `nps_climate_outputs.zip`, in your local repo:

```bash
# Unzip into the site's public folder
unzip nps_climate_outputs.zip -d site/public/data/

# Run the Astro dev server
cd site && npm install && npm run dev
```

Or push to `main` to trigger the GitHub Actions deploy.